# LangGraph Supervisor Fleet — Agent Cost Bleed-Through

## The Problem

The standard LangGraph multi-agent pattern is a supervisor that routes tasks to
specialised sub-agents. When routing logic has a bug — or when the LLM supervisor
decides a task needs multiple agents — the same task flows through several agents
sequentially. Each agent makes its own LLM calls and tool calls.
**Per-task cost multiplies silently.**

One agent misbehaving (stuck in a sub-loop, calling expensive tools repeatedly)
drains the shared budget. The other agents get blocked even though they did nothing wrong.

## The Fix

FiGuard issues a **delegation token** per sub-agent. Each token has a hard cap.
When the researcher blows its $3.00 cap the billing and writer agents are unaffected —
they have their own tokens with their own limits.
The spend tree shows exactly which agent caused the overrun.

In [ ]:
# @title Step 1 — Install and configure FiGuard
!pip install figuard langgraph langchain langchain-openai -q

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
MODE = "simulation"  # Change to "real" to use your own API keys

if MODE == "real":
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")   # Add via Colab Secrets panel (🔑)
    STRIPE_TEST_KEY = userdata.get("STRIPE_TEST_KEY") # Only needed for langchain scenario
else:
    OPENAI_API_KEY = None
    STRIPE_TEST_KEY = None

FIGUARD_BASE_URL = "https://figuard-sandbox-g1ha.onrender.com"

# Note: FiGuard always runs against the live sandbox — real authorization
# decisions and spend tree — even in simulation mode.

import uuid
USER_ID = f"demo_{uuid.uuid4().hex[:8]}"
print(f"Your session ID: {USER_ID}  (identifies your budgets in the dashboard)")


In [ ]:
# ── Display helpers ────────────────────────────────────────────────────────────
def section(title):
    print(f"\n{'═' * 64}")
    print(f"  {title}")
    print(f"{'═' * 64}")

def ok(msg):   print(f"  ✓  {msg}")
def bad(msg):  print(f"  ✗  {msg}")
def info(msg): print(f"     {msg}")
def step(msg): print(f"  →  {msg}")
def agent(name, msg): print(f"  [{name:<12}]  {msg}")

In [ ]:
# ── Scenario tuning ────────────────────────────────────────────────────────────
FLEET_LIMIT      = 10.00   # total fleet budget
BILLING_CAP      = 4.00    # billing agent cap
RESEARCHER_CAP   = 3.00    # researcher cap — this agent overspends in the demo
WRITER_CAP       = 2.00    # writer cap
COST_PER_CALL    = 0.50    # cost per tool call (simulated)
RESEARCHER_CALLS = 9       # 9 × $0.50 = $4.50 → exceeds $3.00 cap at call 7

# ── Sandbox warmup ─────────────────────────────────────────────────────────────
import uuid
from typing import Optional
from figuard import FiGuardClient

figuard = FiGuardClient()
print("Connecting to FiGuard sandbox…", end=" ", flush=True)
run_id = uuid.uuid4().hex[:8]
fleet_budget = figuard.create_budget(
    user_id=USER_ID,
    total_limit=FLEET_LIMIT,
    currency="USD",
    expires_in="1h",
)
fleet_token = fleet_budget.session_token
print("ready.")
print(f"\nMode         : {MODE}")
print(f"Fleet limit  : ${FLEET_LIMIT:.2f}")
print(f"Caps         : billing=${BILLING_CAP:.2f}  researcher=${RESEARCHER_CAP:.2f}  writer=${WRITER_CAP:.2f}")
print(f"Rogue calls  : researcher will attempt {RESEARCHER_CALLS} × ${COST_PER_CALL:.2f} = ${RESEARCHER_CALLS * COST_PER_CALL:.2f}")
print(f"Dashboard    : {FIGUARD_BASE_URL}/ui")

## Part 1 — Without FiGuard

Supervisor routes task to all three agents. The researcher loops on an ambiguous query
with no cap to stop it. All costs are charged to the shared fleet budget with no
per-agent attribution.

In [ ]:
def run_billing_agent(n_calls=2, figuard=None, token=None, label=""):
    """Billing agent: processes invoices. Well-behaved, stays in budget."""
    spent = 0.0
    for i in range(1, n_calls + 1):
        if figuard and token:
            auth = figuard.authorize(
                session_token=token,
                agent_id="billing_agent",
                action_type="INVOICE_PROCESS",
                description=f"Process invoice batch {i}",
                requested_quantity=COST_PER_CALL,
                idempotency_key=f"{label}-billing-{i}",
            )
            if not auth.is_authorized:
                agent("billing", f"call {i} denied — {auth.denial_reason}")
                return i - 1, spent
            agent("billing", f"call {i} authorized (event {auth.event_id[:8]}…)")
            figuard.confirm_event(auth.event_id, confirmed_quantity=COST_PER_CALL)
        else:
            agent("billing", f"call {i} — processing invoice batch")
        spent += COST_PER_CALL
    return n_calls, spent


def run_researcher_agent(n_calls=RESEARCHER_CALLS, figuard=None, token=None, label=""):
    """Researcher agent: makes too many search calls on an ambiguous query."""
    spent = 0.0
    for i in range(1, n_calls + 1):
        if figuard and token:
            auth = figuard.authorize(
                session_token=token,
                agent_id="researcher_agent",
                action_type="SEARCH",
                description=f"Research query iteration {i}",
                requested_quantity=COST_PER_CALL,
                idempotency_key=f"{label}-research-{i}",
            )
            if not auth.is_authorized:
                agent("researcher", f"call {i} denied — {auth.denial_reason}")
                agent("researcher", "cap hit — stopping. other agents unaffected.")
                return i - 1, spent
            agent("researcher", f"call {i} authorized (event {auth.event_id[:8]}…)")
            figuard.confirm_event(auth.event_id, confirmed_quantity=COST_PER_CALL)
        else:
            agent("researcher", f"call {i} — searching…")
        spent += COST_PER_CALL
    return n_calls, spent


def run_writer_agent(n_calls=2, figuard=None, token=None, label=""):
    """Writer agent: generates report. Well-behaved."""
    spent = 0.0
    for i in range(1, n_calls + 1):
        if figuard and token:
            auth = figuard.authorize(
                session_token=token,
                agent_id="writer_agent",
                action_type="GENERATE",
                description=f"Report section {i}",
                requested_quantity=COST_PER_CALL,
                idempotency_key=f"{label}-writer-{i}",
            )
            if not auth.is_authorized:
                agent("writer", f"call {i} denied — {auth.denial_reason}")
                return i - 1, spent
            agent("writer", f"call {i} authorized (event {auth.event_id[:8]}…)")
            figuard.confirm_event(auth.event_id, confirmed_quantity=COST_PER_CALL)
        else:
            agent("writer", f"call {i} — writing report section")
        spent += COST_PER_CALL
    return n_calls, spent


section("PART 1 — Without FiGuard")
info("Supervisor routes task to all three agents.")
info("Researcher loops on ambiguous query — no cap stops it.")
info("All costs charged to the shared fleet budget.\n")

b_calls, b_cost = run_billing_agent(n_calls=2)
r_calls, r_cost = run_researcher_agent()
w_calls, w_cost = run_writer_agent(n_calls=2)

total = b_cost + r_cost + w_cost
print()
info(f"  billing    : {b_calls} calls  ${b_cost:.2f}")
info(f"  researcher : {r_calls} calls  ${r_cost:.2f}  ← runaway")
info(f"  writer     : {w_calls} calls  ${w_cost:.2f}")
bad(f"Total fleet cost: ${total:.2f}  (budget was ${FLEET_LIMIT:.2f})")
bad("No per-agent visibility. No way to know which agent overspent.")

## Part 2 — With FiGuard Delegation Tokens

Same fleet budget. Each agent gets a delegation token with a hard cap.
The researcher hits its $3.00 cap. Billing and writer are unaffected —
they have independent tokens.

In [ ]:
from figuard import clear_current_event_id
section("PART 2 — With FiGuard delegation tokens")
info("Same fleet budget. Each agent gets a delegation token with a hard cap.")
info("Researcher hits its cap. Billing and writer are unaffected.\n")

# Issue one delegation token per sub-agent.
billing_token = figuard.create_delegation_token(
    budget_id=fleet_budget.id,
    label="billing-agent",
    caps=[{"category": "billing", "limit": BILLING_CAP}],
)
researcher_token = figuard.create_delegation_token(
    budget_id=fleet_budget.id,
    label="researcher-agent",
    caps=[{"category": "research", "limit": RESEARCHER_CAP}],
)
writer_token = figuard.create_delegation_token(
    budget_id=fleet_budget.id,
    label="writer-agent",
    caps=[{"category": "writer", "limit": WRITER_CAP}],
)

ok(f"Fleet budget : {fleet_budget.id}")
ok(f"billing      : cap ${BILLING_CAP:.2f}  token {billing_token.session_token[:12]}…")
ok(f"researcher   : cap ${RESEARCHER_CAP:.2f}  token {researcher_token.session_token[:12]}…")
ok(f"writer       : cap ${WRITER_CAP:.2f}  token {writer_token.session_token[:12]}…")
info("")

clear_current_event_id()
b_calls, b_cost = run_billing_agent(
    n_calls=2, figuard=figuard,
    token=billing_token.session_token, label=run_id,
)
clear_current_event_id()
r_calls, r_cost = run_researcher_agent(
    figuard=figuard,
    token=researcher_token.session_token, label=run_id,
)
clear_current_event_id()
w_calls, w_cost = run_writer_agent(
    n_calls=2, figuard=figuard,
    token=writer_token.session_token, label=run_id,
)

total_guarded = b_cost + r_cost + w_cost
print()
ok(f"  billing    : {b_calls} calls  ${b_cost:.2f}  — completed")
bad(f"  researcher : {r_calls} calls  ${r_cost:.2f}  "
    f"— capped at ${RESEARCHER_CAP:.2f} (saved "
    f"${(RESEARCHER_CALLS * COST_PER_CALL - r_cost):.2f})")
ok(f"  writer     : {w_calls} calls  ${w_cost:.2f}  — completed")
ok(f"Total fleet cost: ${total_guarded:.2f}  (saved ${(total - total_guarded):.2f} vs unguarded)")

## What FiGuard Recorded

Open the dashboard and find the budget printed above.
Go to **Budget → Spend Tree** to see the per-agent causal chain.
Three sub-trees — one per agent. The researcher sub-tree ends with DENIED.

In [ ]:
b = figuard.get_budget(fleet_budget.id)

section("What FiGuard recorded")
info(f"Dashboard : {FIGUARD_BASE_URL}/ui/budgets/{fleet_budget.id}")
info(f"Budget    : {fleet_budget.id}")
info("")
ok(f"Fleet spent : ${b.quantity_spent:.2f} of ${FLEET_LIMIT:.2f}")
ok(f"Remaining   : ${b.available_quantity:.2f}")
info("")
info("Open the budget → Spend Tree.")
info("You'll see three sub-trees — one per agent.")
info("The researcher sub-tree ends with DENIED / DELEGATE_CAP_EXCEEDED.")
info("Billing and writer sub-trees end with CONFIRMED.")
info("")
info("Without delegation tokens: one flat ledger, no attribution.")
info("With delegation tokens: per-agent spend is visible and bounded.")
print(f"\n  Dashboard link: {FIGUARD_BASE_URL}/ui/budgets/{fleet_budget.id}")